# 04. Logistic Regression Model Training — Baseline (no preprocessing)

This notebook trains a **Logistic Regression** classifier with **default settings** on all **103 signal features**, without any preprocessing, to get a first baseline score.

It uses exactly the same data and steps as `04d_lightgbm_baseline.ipynb`, so the two results can be compared.

Important difference from LightGBM:
- Logistic Regression is **sensitive to feature scale**. Features with large values (for example power or energy features) can dominate the model, and the solver may need more iterations to converge.
- No scaling is done here **on purpose**: this run shows how Logistic Regression performs *without* preprocessing.
- If a `ConvergenceWarning` appears, it is a real result of this experiment and is **not hidden**.

Rules followed:
- Train on `train.csv` only, score on `val.csv` only.
- `test.csv` is **not loaded** in this notebook.
- `record_id`, `patient_id` and `figo_label` are not used as input features.
- Positive class = abnormal (`figo_label = 1`). Threshold = 0.5 (default).

## Step 1: Setup and imports

In [1]:
import sys
sys.path.append('../..')

import time
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import f1_score, roc_auc_score, balanced_accuracy_score, confusion_matrix

from config import config
from src.data_preprocessing import get_feature_columns

## Step 2: Load the training and validation sets

The test set stays untouched until the final evaluation.

In [2]:
# Load train and validation sets only (test set stays untouched)
train_df = pd.read_csv(config.PROCESSED_TRAIN_FILE)
val_df = pd.read_csv(config.PROCESSED_VAL_FILE)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)

Train shape: (14540, 106)
Validation shape: (3115, 106)


## Step 3: Separate features (X) and target (y)

In [3]:
# Get the 103 signal features (IDs and target are excluded)
feature_cols = get_feature_columns(train_df)

X_train = train_df[feature_cols]
y_train = train_df[config.TARGET_COLUMN]

X_val = val_df[feature_cols]
y_val = val_df[config.TARGET_COLUMN]

print("Train:", X_train.shape, " Validation:", X_val.shape)

Number of feature columns found: 103
Train: (14540, 103)  Validation: (3115, 103)


## Step 4: Create and train the Logistic Regression model

Default hyperparameters (solver `lbfgs`, `max_iter=100`); the random seed is fixed for consistency with the other notebooks.

In [4]:
# Create Logistic Regression with default settings (only the seed is fixed)
model = LogisticRegression(random_state=config.RANDOM_STATE)

# Train on the training set only
start_time = time.time()
model.fit(X_train, y_train)
train_seconds = time.time() - start_time

print("Training time (seconds):", round(train_seconds, 2))

# Number of iterations the solver used (if it equals 100, it stopped at the limit)
print("Solver iterations used:", model.n_iter_[0])

Training time (seconds): 2.1
Solver iterations used: 100


c:\Users\Rathishan\Desktop\Desktop\Git\ML_Module\Machine-Learning-Based-Fetal-Well-being-Classification-Using-Cardiotocography-Features\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Step 5: Predict on the validation set

In [5]:
# Class predictions (threshold 0.5 by default)
val_pred = model.predict(X_val)

# Probability of abnormal (1), needed for ROC-AUC
val_prob = model.predict_proba(X_val)[:, 1]

## Step 6: Validation metrics

Positive class = abnormal (`figo_label = 1`).

In [6]:
print("Accuracy:         ", round(accuracy_score(y_val, val_pred), 4))
print("Precision:        ", round(precision_score(y_val, val_pred), 4))
print("Recall:           ", round(recall_score(y_val, val_pred), 4))
print("F1-score:         ", round(f1_score(y_val, val_pred), 4))
print("ROC-AUC:          ", round(roc_auc_score(y_val, val_prob), 4))
print("Balanced accuracy:", round(balanced_accuracy_score(y_val, val_pred), 4))

print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_val, val_pred))

Accuracy:          0.713
Precision:         0.742
Recall:            0.7154
F1-score:          0.7284
ROC-AUC:           0.7863
Balanced accuracy: 0.7128
Confusion matrix [[TN, FP], [FN, TP]]:
[[1022  417]
 [ 477 1199]]


## Step 7: Check for overfitting

If the training F1 is much higher than the validation F1, the model is overfitting.

In [7]:
# Training F1, compared with validation F1 from Step 6
train_pred = model.predict(X_train)
print("Training F1:  ", round(f1_score(y_train, train_pred), 4))
print("Validation F1:", round(f1_score(y_val, val_pred), 4))

Training F1:   0.729
Validation F1: 0.7284
